# Privacy Demo
This notebook demonstrates simple privacy-preserving techniques (e.g., masking, aggregation, differential privacy examples) on sample data.

Sections:
- Sample data generation
- Masking and redaction
- Aggregation and k-anonymity checks
- (Optional) Differential privacy sketch

In [ ]:
# Placeholder: import libraries
import pandas as pd

# TODO: add privacy-preserving examples

# Privacy Risk Assessment — NovaCred Credit Applications

## Context

NovaCred is a fintech company using data-driven models to evaluate credit applications.  
Because these systems process large amounts of **personal and financial data**, strong data governance and privacy controls are required.

This notebook examines the credit application dataset to identify **privacy risks and governance gaps**.

The analysis focuses on:

- Identifying personally identifiable information (PII)
- Evaluating potential re-identification risks
- Demonstrating pseudonymization techniques
- Linking findings to relevant **GDPR principle**

The goal is to show how privacy-preserving practices can be integrated into a credit decision data pipeline.

## 1. Dataset Loading

In [1]:
import pandas as pd
import numpy as np
import json

In [5]:
# Load raw dataset
with open("../data/data.json", "r") as f:
    data = json.load(f)

print("Number of credit applications:", len(data))

Number of credit applications: 502


In [6]:
df = pd.json_normalize(data)

df.head()

,_id,spending_behavior,processing_timestamp,applicant_info.full_name,applicant_info.email,applicant_info.ssn,applicant_info.ip_address,applicant_info.gender,applicant_info.date_of_birth,applicant_info.zip_code,...,financials.credit_history_months,financials.debt_to_income,financials.savings_balance,decision.loan_approved,decision.rejection_reason,loan_purpose,decision.interest_rate,decision.approved_amount,financials.annual_salary,notes
0,app_200,"[{'category': 'Shopping', 'amount': 480}, {'ca...",2024-01-15T00:00:00Z,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,...,23,0.20,31212,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
1,app_037,"[{'category': 'Rent', 'amount': 608}, {'catego...",NaN,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,...,51,0.18,17915,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN
2,app_215,"[{'category': 'Rent', 'amount': 109}]",NaN,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,...,41,0.21,37909,True,NaN,vacation,3.7,59000.0,NaN,NaN
3,app_024,"[{'category': 'Fitness', 'amount': 575}]",NaN,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077,...,70,0.35,0,True,NaN,NaN,4.3,34000.0,NaN,NaN
4,app_184,"[{'category': 'Entertainment', 'amount': 463}]",2024-01-15T00:00:00Z,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,M,1999-05-21,10080,...,14,0.23,31763,False,algorithm_risk_score,NaN,NaN,NaN,NaN,NaN


In [7]:
df.columns

Index(['_id', 'spending_behavior', 'processing_timestamp',
       'applicant_info.full_name', 'applicant_info.email',
       'applicant_info.ssn', 'applicant_info.ip_address',
       'applicant_info.gender', 'applicant_info.date_of_birth',
       'applicant_info.zip_code', 'financials.annual_income',
       'financials.credit_history_months', 'financials.debt_to_income',
       'financials.savings_balance', 'decision.loan_approved',
       'decision.rejection_reason', 'loan_purpose', 'decision.interest_rate',
       'decision.approved_amount', 'financials.annual_salary', 'notes'],
      dtype='object')

## 2. PII Identification

To evaluate privacy risks, we first identify fields that contain **personally identifiable information (PII)** or sensitive personal data.

These fields may allow **direct identification** of individuals or increase the risk of **re-identification when combined with other attributes**.

### 2.1 PII Fields in the Dataset

The dataset contains several attributes that can directly or indirectly identify individuals.

These fields represent potential privacy risks and must be carefully handled in data governance and regulatory compliance contexts.

In [9]:
pii_fields = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.gender",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code"
]

print("Potential PII fields identified:\n")

for field in pii_fields:
    print("-", field)

Potential PII fields identified:

- applicant_info.full_name
- applicant_info.email
- applicant_info.ssn
- applicant_info.ip_address
- applicant_info.gender
- applicant_info.date_of_birth
- applicant_info.zip_code


### 2.2 PII Risk Classification

After identifying potential PII fields, we classify them based on their **identification risk** and **privacy sensitivity**.

We distinguish between:

- **Direct identifiers** – uniquely identify an individual (e.g., name, SSN)
- **Quasi-identifiers** – may identify individuals when combined with other attributes
- **Sensitive attributes** – data that may reveal personal behavior or protected characteristics

In [12]:
pii_classification = {
    "applicant_info.full_name": "Direct identifier",
    "applicant_info.email": "Direct identifier",
    "applicant_info.ssn": "Highly sensitive identifier",
    "applicant_info.ip_address": "Online identifier",
    "applicant_info.date_of_birth": "Quasi identifier",
    "applicant_info.zip_code": "Quasi identifier",
    "applicant_info.gender": "Protected attribute",
    "spending_behavior": "Sensitive behavioral data"
}

pd.DataFrame.from_dict(
    pii_classification,
    orient="index",
    columns=["PII Category"]
)

,PII Category
applicant_info.full_name,Direct identifier
applicant_info.email,Direct identifier
applicant_info.ssn,Highly sensitive identifier
applicant_info.ip_address,Online identifier
applicant_info.date_of_birth,Quasi identifier
applicant_info.zip_code,Quasi identifier
applicant_info.gender,Protected attribute
spending_behavior,Sensitive behavioral data


These classifications highlight the privacy risks associated with different attributes in the dataset. Direct identifiers such as names, email addresses, and social security numbers pose the highest identification risk because they can uniquely identify an individual.

Quasi-identifiers such as ZIP code and date of birth may not identify individuals on their own, but they can enable re-identification when combined with other attributes.

Behavioral attributes such as `spending_behavior` can also reveal sensitive lifestyle and financial patterns. Even though they are not direct identifiers, they may allow inference of personal characteristics and therefore require careful governance and protection.

This demonstrates the importance of applying privacy-preserving techniques such as pseudonymization or aggregation when handling personal data in credit decision systems.

## 3. Pseudonymization

To reduce privacy risks, direct identifiers can be replaced with pseudonymous values. 
Pseudonymization allows data to be used for analysis while protecting the original identity of individuals.

In this example, we apply hashing to email addresses to replace the original identifier with a pseudonymous value.

In [13]:
import hashlib
import pandas as pd

# Function to pseudonymize email using SHA-256
def hash_email(email):
    if pd.isna(email):
        return email
    return hashlib.sha256(email.encode()).hexdigest()

# Create pseudonymized column
df["email_hash"] = df["applicant_info.email"].apply(hash_email)

# Show original vs pseudonymized values
print("Sample of original vs pseudonymized email values:\n")
display(df[["applicant_info.email", "email_hash"]].head())

# Data minimization: remove original identifier
df_privacy = df.drop(columns=["applicant_info.email"])

print("\nOriginal column count:", df.shape[1])
print("Column count after removing email:", df_privacy.shape[1])
print("Original email column removed for privacy protection.")

Sample of original vs pseudonymized email values:



,applicant_info.email,email_hash
0,jerry.smith17@hotmail.com,116648a7761525746032d0ab323ceb01f50d11f7935164...
1,brandon.walker2@yahoo.com,c3522c0b54ef9045c73186bcabb53f8e512360ed17e9cc...
2,scott.moore94@mail.com,b299e7d6a37e183bab209eb8df919652117dd16ed16698...
3,thomas.lee6@protonmail.com,6fbd2478748a29faa143392f28d955133e346d09673963...
4,brian.rodriguez86@aol.com,f24e7cc1450ee9aa26b833e0593b2420fbfd59b8ad2636...



Original column count: 22
Column count after removing email: 21
Original email column removed for privacy protection.


The pseudonymization process replaces direct identifiers with hashed values, reducing the risk of exposing personal information.

Although the original email addresses are no longer directly visible, the pseudonymized identifiers can still be used for analytical tasks such as grouping records or tracking behavioral patterns.

This approach helps balance data utility with privacy protection in data-driven decision systems.

This example also demonstrates the GDPR principle of data minimization, where unnecessary direct identifiers are removed after pseudonymization.

## 4. GDPR Mapping

The identified dataset attributes and processing steps can be mapped to relevant GDPR principles and regulatory requirements. This helps clarify which legal obligations apply to the credit scoring system.

In [20]:
gdpr_mapping = {
    "Art. 6 — Lawful Basis":
    "Credit scoring must rely on a lawful basis such as contractual necessity when processing loan applications.",

    "Art. 5 — Data Minimization":
    "Only necessary attributes should be processed. Direct identifiers such as applicant_info.email should be removed after analysis.",

    "Art. 4(1) — Personal Data":
    "Attributes like applicant_info.full_name, applicant_info.email, IP address, ZIP code, and date_of_birth qualify as personal data.",

    "Art. 4(5) — Pseudonymization":
    "Email addresses are pseudonymized using SHA-256 hashing and the original identifier is removed from the analytical dataset.",

    "Art. 22 — Automated Decision-Making":
    "Credit scoring may involve automated decisions with significant effects, requiring transparency and safeguards.",

    "Art. 32 — Security of Processing":
    "Sensitive identifiers such as SSN require strong protection through hashing, encryption, and access control."
}

pd.DataFrame(
    list(gdpr_mapping.items()),
    columns=["GDPR Article", "Application in the Dataset"]
).style.hide(axis="index")

Matplotlib is building the font cache; this may take a moment.


GDPR Article,Application in the Dataset
Art. 6 — Lawful Basis,Credit scoring must rely on a lawful basis such as contractual necessity when processing loan applications.
Art. 5 — Data Minimization,Only necessary attributes should be processed. Direct identifiers such as applicant_info.email should be removed after analysis.
Art. 4(1) — Personal Data,"Attributes like applicant_info.full_name, applicant_info.email, IP address, ZIP code, and date_of_birth qualify as personal data."
Art. 4(5) — Pseudonymization,Email addresses are pseudonymized using SHA-256 hashing and the original identifier is removed from the analytical dataset.
Art. 22 — Automated Decision-Making,"Credit scoring may involve automated decisions with significant effects, requiring transparency and safeguards."
Art. 32 — Security of Processing,"Sensitive identifiers such as SSN require strong protection through hashing, encryption, and access control."


This mapping demonstrates how the dataset attributes relate to key GDPR provisions. 
Several variables in the dataset qualify as personal data under Article 4(1), including 
direct identifiers such as name and email as well as quasi-identifiers like ZIP code 
and date of birth.

To reduce privacy risks, pseudonymization and data minimization techniques were applied. 
Direct identifiers such as email addresses were replaced with hashed values, and the 
original identifier was removed from the analytical dataset. This approach reduces the 
risk of re-identification while still allowing the data to be used for analytical tasks 
such as credit risk assessment.

However, because the system may support automated decision-making in credit scoring, 
additional safeguards should be implemented to ensure compliance with GDPR. These include 
transparency about automated decisions, appropriate security measures for sensitive 
identifiers such as SSN, and clear governance procedures for handling personal data.